<h2>Why Use SQLite?</h2>

<p>
SQLite is a lightweight, file-based database that allows you to store and query structured data efficiently.  
For this project, it provides several advantages:
</p>

<ul>
  <li><strong>Simple setup:</strong> No need to install or configure a separate database server.</li>
  <li><strong>Portable:</strong> The entire database is stored in a single <code>.db</code> file, making it easy to move or share.</li>
  <li><strong>Good for small to medium datasets:</strong> Our profiles and titles tables are small enough that SQLite can handle them efficiently.</li>
  <li><strong>Full SQL support:</strong> You can run queries, joins, and filters directly on the database without loading everything into memory.</li>
</ul>

<h3>Database Design</h3>
<p>
The plan is to create a database file called <code>streamly.db</code> with three tables:
</p>

<ul>
  <li><strong>accounts:</strong> inferred from <code>profiles.account_id</code>, one row per account.</li>
  <li><strong>profiles:</strong> one row per profile, storing user details and preferences.</li>
  <li><strong>titles:</strong> one row per title, storing show information like category, rating, and language.</li>
</ul>

<h3>Loading Data into SQLite</h3>
<p>
First, the cleaned CSV files are read into pandas DataFrames
</p>

<p>
Once loaded, these DataFrames can be inserted into the SQLite database, allowing SQL queries and easy integration with the recommendation engine.
</p>

In [1]:
import sqlite3
import pandas as pd

profiles = pd.read_csv("../data/profiles_clean.csv")
titles = pd.read_csv("../data/titles_clean.csv")

<h2>Inferring Account Plans</h2>

<p>
Each account may have one or more profiles associated with it.  
To better understand account types, a <strong>plan type</strong> is inferred based on the number of profiles under each account.
</p>

<h3>Counting Profiles per Account</h3>
<p>
The first step is to count how many profiles belong to each account.  
This allows us to see whether an account has just one profile or multiple profiles.
</p>

<h3>Inferring Plan Type</h3>
<p>
The number of profiles is then mapped to a plan category:
</p>
<ul>
  <li>1 profile or fewer → <strong>Basic</strong></li>
  <li>2–4 profiles → <strong>Premium</strong></li>
  <li>More than 4 profiles → <strong>Premium-Plus</strong> (used as a safety check for unexpected cases)</li>
</ul>

<h3>Creating Accounts Table</h3>
<p>
A new <strong>accounts</strong> table is created using the profile counts and plan types.  
The earliest profile creation date within each account is also recorded to track when the account was first created.
</p>

<h3>Resulting Table</h3>
<p>
The accounts table now contains the following information:
</p>
<ul>
  <li><strong>account_id:</strong> unique identifier for each account</li>
  <li><strong>profile_count:</strong> number of profiles under the account</li>
  <li><strong>plan_type:</strong> inferred plan based on profile count</li>
  <li><strong>created_at:</strong> earliest profile creation date</li>
</ul>

<p>
This table can now be used for filtering or segmenting users based on their account plan type and for insertion into the SQLite database.
</p>

In [4]:
account_counts = profiles.groupby("account_id")["profile_id"].count().reset_index(name = "profile_count")

def infer_plan_type(count):
    if count <= 1:
        return "Basic"
    elif count <= 4:
        return "Pemium"
    else:
        return "Premium-Plus" # safety for unexpected cases
    
accounts = account_counts.copy()
accounts["plan_type"] = accounts["profile_count"].apply(infer_plan_type)
accounts["created_at"] = pd.to_datetime(profiles.groupby("account_id")["created_at"].min()).values
accounts.head()

,account_id,profile_count,plan_type,created_at
0,1,1,Basic,2022-10-21
1,2,1,Basic,2022-06-12
2,3,1,Basic,2023-02-20
3,4,4,Pemium,2022-04-18
4,5,2,Pemium,2022-09-07


<h1> Create DB & tables </h1>

<h2>Creating SQLite Tables</h2>

<p>
A SQLite database named <strong>streamly.db</strong> is created with three tables: <strong>accounts</strong>, <strong>profiles</strong>, and <strong>titles</strong>.  
Existing tables are dropped first to allow reruns without errors.
</p>

<ul>
  <li><strong>Accounts:</strong> stores account information including plan type, number of profiles, and creation date.</li>
  <li><strong>Profiles:</strong> stores one row per profile, with a link to the corresponding account and a list of preferences.</li>
  <li><strong>Titles:</strong> stores one row per title, including category, duration, age rating, language, and popularity metrics.</li>
</ul>

<p>
Foreign keys ensure that profiles are linked to accounts, maintaining relational integrity.
</p>

In [5]:
conn = sqlite3.connect("../streamly.db")
cur = conn.cursor()

# drop if exists for reruns

cur.execute("DROP TABLE IF EXISTS profiles;")
cur.execute("DROP TABLE IF EXISTS titles;")
cur.execute("DROP TABLE IF EXISTS accounts;")

cur.execute("""
CREATE TABLE accounts (
     account_id INTEGER PRIMARY KEY,
     plan_type TEXT,
     profile_count INTEGER,
     created_at TEXT
);            
""")

cur.execute("""
CREATE TABLE profiles (
    profile_id INTEGER PRIMARY KEY,
    account_id INTEGER,
    profile_name TEXT,
    kids_profile INTEGER,
    age_band TEXT,
    preferred_language TEXT,
    created_at TEXT,
    preferences TEXT,
    preference_list TEXT,        
    FOREIGN KEY(account_id) REFERENCES accounts(account_id)
);
""")

cur.execute("""
CREATE TABLE titles(
    show_id TEXT PRIMARY KEY,
    title_name TEXT,
    category TEXT,
    sub_category TEXT,
    duration REAL,
    age_rating TEXT,
    type TEXT,
    year INTEGER,
    origin_region TEXT,
    language TEXT,
    episode_count INTEGER,
    is_kids_content INTEGER,
    imdb_rating REAL,
    imdb_votes INTEGER
            );
""")

conn.commit();

<h2>Inserting Data into SQLite</h2>

<p>
After creating the tables, the cleaned data from the <strong>accounts</strong>, <strong>profiles</strong>, and <strong>titles</strong> DataFrames is inserted into the SQLite database.
</p>

<ul>
  <li><strong>Accounts:</strong> inserted with account ID, plan type, profile count, and creation date.</li>
  <li><strong>Profiles:</strong> inserted with profile details, including a numeric flag for kids profiles and a parsed list of preferences.</li>
  <li><strong>Titles:</strong> inserted with title information, converting the kids content flag to numeric for consistency.</li>
</ul>

<p>
All changes are committed to the database, making the data ready for querying and the recommendation engine.
</p>

In [6]:

# insert accounts
accounts_to_insert = accounts[["account_id", "plan_type", "profile_count", "created_at"]].copy()
accounts_to_insert["created_at"] = accounts_to_insert["created_at"].astype(str)

accounts_to_insert.to_sql("accounts", conn, if_exists="append", index=False)

# insert profiles
profiles_to_insert = profiles.copy()
profiles_to_insert["kids_profile"] = profiles_to_insert["kids_profile"].astype(int)
profiles_to_insert["created_at"] = pd.to_datetime(profiles_to_insert["created_at"]).astype(str)

profiles_to_insert.to_sql("profiles", conn, if_exists="append", index=False)

# insert titles
titles_to_insert = titles.copy()
titles_to_insert["is_kids_content"] = titles_to_insert["is_kids_content"].astype(int)

titles_to_insert.to_sql("titles", conn, if_exists="append", index=False)

conn.commit()


<h5> Sanity checks </h5>


In [7]:
print("Accounts:", pd.read_sql_query("SELECT COUNT(*) AS n FROM accounts;", conn))
print("Profiles:", pd.read_sql_query("SELECT COUNT(*) AS n FROM profiles;", conn))
print("Titles:", pd.read_sql_query("SELECT COUNT(*) AS n FROM titles;", conn))

display(pd.read_sql_query("""
SELECT p.profile_id, p.profile_name, a.plan_type
FROM profiles p
JOIN accounts a ON p.account_id = a.account_id
LIMIT 10;                                                                     
""", conn))

conn.close()


Accounts:      n
0  200
Profiles:      n
0  359
Titles:       n
0  1000


,profile_id,profile_name,plan_type
0,1,Gran,Basic
1,2,Vô,Basic
2,3,Sam,Basic
3,4,Vó,Pemium
4,5,Pai,Pemium
5,6,Mãe,Pemium
6,7,Vô,Pemium
7,8,Abuela,Pemium
8,9,Guest,Pemium
9,10,Jordan,Pemium
